# COPC Conversion

Convert LAS/LAZ point cloud files to **COPC** (Cloud-Optimized Point Cloud) format.

## What is COPC?

COPC is a standard LAZ 1.4 file with an embedded spatial octree index. This enables:

- **Efficient streaming**: HTTP range requests load only the spatial region needed
- **Fast partial reads**: skip loading the entire file for local visualization
- **Web compatibility**: serve directly to Potree, copc.io viewer, and QGIS 3.26+

COPC files use the `.copc.laz` extension and are fully compatible with all LAS/LAZ tools.

**Requires**: PDAL (included in Docker image and conda environment)

In [ ]:
from pathlib import Path
from sat.utils.paths import get_output_dir

# Auto-detect paths (works in Docker and local installs)
results_dir = get_output_dir() / 'final_results'
print(f"Results directory: {results_dir}")
if not results_dir.exists():
    print(f"WARNING: {results_dir} does not exist. Run inference first.")

In [ ]:
# Verify PDAL is available
import subprocess
result = subprocess.run(['pdal', '--version'], capture_output=True, text=True)
print(result.stdout or result.stderr)

## Convert a Single File

In [ ]:
from sat.io.las_io import laz_to_copc

# Find a LAZ file to convert (skip files already in COPC format)
laz_files = [f for f in results_dir.glob('*.laz') if '.copc.laz' not in f.name] if results_dir.exists() else []

if laz_files:
    input_file = str(laz_files[0])
    output_file = input_file.replace('.laz', '.copc.laz')
    laz_to_copc(input_file, output_file, verbose=True)
    print(f"\nOriginal: {Path(input_file).stat().st_size / 1e6:.1f} MB")
    print(f"COPC:     {Path(output_file).stat().st_size / 1e6:.1f} MB")
else:
    print("No unconverted LAZ files found.")
    print(f"Either run inference first, or check {results_dir} for existing files.")
    copc_files = list(results_dir.glob('*.copc.laz')) if results_dir.exists() else []
    if copc_files:
        print(f"Found {len(copc_files)} existing COPC files (already converted).")

## Batch Convert All LAZ Files

In [ ]:
from sat.io.las_io import laz_to_copc

laz_files = [f for f in results_dir.glob('*.laz') if '.copc.laz' not in f.name] if results_dir.exists() else []
print(f"Found {len(laz_files)} LAZ files to convert")

for laz_file in laz_files:
    copc_file = str(laz_file).replace('.laz', '.copc.laz')
    if Path(copc_file).exists():
        print(f"  Skip (exists): {laz_file.name}")
        continue
    try:
        laz_to_copc(str(laz_file), copc_file, verbose=True)
    except Exception as e:
        print(f"  Failed: {laz_file.name} — {e}")

print(f"\nCOPC files:")
for f in sorted(results_dir.glob('*.copc.laz')) if results_dir.exists() else []:
    print(f"  {f.name} ({f.stat().st_size / 1e6:.1f} MB)")

## Verify COPC with PDAL

In [ ]:
import subprocess, json

copc_files = sorted(results_dir.glob('*.copc.laz')) if results_dir.exists() else []

for f in copc_files[:3]:  # Check first 3
    result = subprocess.run(
        ['pdal', 'info', '--summary', str(f)],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        info = json.loads(result.stdout)
        summary = info.get('summary', {})
        bounds = summary.get('bounds', {})
        print(f"{f.name}:")
        print(f"  Points: {summary.get('num_points', 'N/A'):,}")
        print(f"  Dimensions: {summary.get('num_dims', 'N/A')}")
        if bounds:
            print(f"  Bounds X: [{bounds.get('minx', 'N/A'):.1f}, {bounds.get('maxx', 'N/A'):.1f}]")
            print(f"  Bounds Y: [{bounds.get('miny', 'N/A'):.1f}, {bounds.get('maxy', 'N/A'):.1f}]")
        print()
    else:
        print(f"{f.name}: PDAL error — {result.stderr}")

if not copc_files:
    print("No COPC files found. Run conversion first.")

## Serving COPC Files for Web Viewing

COPC files can be served over HTTP and visualized in a browser:

1. **Start a local HTTP server** in the output directory:
   ```bash
   cd /path/to/output/final_results
   python -m http.server 8080
   ```

2. **Open the [copc.io viewer](https://viewer.copc.io/)** and enter your file URL:
   ```
   http://localhost:8080/your_file.copc.laz
   ```

3. For **public sharing**, upload your COPC file to any HTTP-accessible storage (S3, GCS, or CyVerse Data Store) and share the direct URL.

The spatial octree index in COPC files means viewers only download the tiles needed for the current view — no need to transfer the entire file.